# 00 — Smoke Test: MiDaS Inference

Verify that:
- The environment is set up correctly
- MiDaS `dpt_hybrid_384` loads and runs
- GPU is detected (if available)
- `.npy` predictions are saved to `outputs/predictions/smoke/`

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import glob

from src.adapters.midas_adapter import MiDaSAdapter

adapter = MiDaSAdapter(model_type='dpt_hybrid_384')

In [ ]:
input_dir = PROJECT_ROOT / 'third_party' / 'MiDaS' / 'input'
image_paths = sorted(
    glob.glob(str(input_dir / '*.jpg')) +
    glob.glob(str(input_dir / '*.png'))
)
print(f'Found {len(image_paths)} images in {input_dir}')
for p in image_paths:
    print(' ', p)

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'predictions' / 'smoke'
records = adapter.run_batch(image_paths, output_dir=str(OUTPUT_DIR), verbose=True)
print(f'\n{sum(r["error"] is None for r in records)}/{len(records)} succeeded')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

ok = [r for r in records if r['error'] is None]
n_show = min(4, len(ok))

fig, axes = plt.subplots(n_show, 2, figsize=(14, 4 * n_show))
if n_show == 1:
    axes = [axes]

for i, rec in enumerate(ok[:n_show]):
    img = cv2.cvtColor(cv2.imread(rec['image_path']), cv2.COLOR_BGR2RGB)
    depth = np.load(rec['pred_path'])

    axes[i][0].imshow(img)
    axes[i][0].set_title(Path(rec['image_path']).name)
    axes[i][0].axis('off')

    axes[i][1].imshow(depth, cmap='inferno')
    axes[i][1].set_title(f'Depth  shape={depth.shape}  runtime={rec["runtime_s"]*1000:.0f}ms')
    axes[i][1].axis('off')

plt.tight_layout()
plt.show()

## Smoke test passed

If you see depth maps above, the pipeline is working.

Next step: run `notebooks/01_build_kittic_manifest.ipynb` after downloading KITTI-C.